# Rapprochement DPE aout 2026

In [5]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [6]:
usecols = ['numero_dpe', 'identifiant_ban', 'code_postal_ban', 'code_postal_brut', 'id_rnb', 'provenance_id_rnb']
dtype={'numero_dpe': 'string', 'identifiant_ban': 'string', 'code_postal_ban': 'string', 'code_postal_brut': 'string', 'id_rnb': 'string', 'provenance_id_rnb': 'string'}

# usecols = ['numero_dpe', 'id_rnb', 'provenance_id_rnb']
# dtype={'numero_dpe': 'string', 'id_rnb': 'string', 'provenance_id_rnb': 'string'}

In [3]:
df_tertiaire = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe01tertiaire.csv', sep=',', usecols=usecols, dtype=dtype)
df_tertiaire['file'] = 'tertiaire'

In [4]:
df_existant = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe03existant.csv', sep=',', usecols=usecols, dtype=dtype)
df_existant['file'] = 'existant'

KeyboardInterrupt: 

In [ ]:
df_neuf = pd.read_csv('notebooks/rapprochements/DPE/2026/data/dpe02neuf.csv', sep=',', usecols=usecols, dtype=dtype)
df_neuf['file'] = 'neuf'

In [ ]:
df = pd.concat([df_tertiaire, df_neuf, df_existant])

In [ ]:
df.shape

(17255757, 7)

In [ ]:
df.head()

,numero_dpe,id_rnb,provenance_id_rnb,code_postal_ban,identifiant_ban,code_postal_brut,file
0,2618T0063102M,<NA>,<NA>,18200,18197_0540,18200,tertiaire
1,2606T0031071B,<NA>,<NA>,06210,06079_0602_00026,06210,tertiaire
2,2665T0010661I,<NA>,<NA>,65000,65440_1990_00040,65000,tertiaire
3,2692T0076695D,<NA>,<NA>,92000,92050_4448,92000,tertiaire
4,2675T0079080F,8PAF1D23GEJC,Logiciel,75019,75119_8241_00058,75020,tertiaire


In [ ]:
# create 100 subfiles in the dpe_existant folder, using the number of rows in the dataframe
for i in range(0, 100):
    df.iloc[i*df.shape[0]//100:(i+1)*df.shape[0]//100].to_csv(f'notebooks/rapprochements/DPE/2026/sub_files/dpe-{i}.csv', index=False)

In [7]:
# le rapprochement

import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
from django.db import connection
from concurrent.futures import ThreadPoolExecutor
import numpy as np

SQL = """
        with rnb_ids as (
        select
            rnb_id
        from
            batid_buildingaddressesreadonly bb
        left join batid_building bb2 on
            bb2.id = bb.building_id
        where
            address_id = %s
            and ST_AREA(shape::geography) > 25
            and bb2.is_active)
        select
            array_agg(rnb_id)
        from
            rnb_ids;
"""

def normalize_postal_code(code):
    # in the "neuf" and "existant" source files the postal code is stored as a
    # number, so a leading zero is missing ("1480" instead of "01480")
    if pd.isna(code):
        return None
    return str(code).strip().zfill(5)

def same_postal_code(row):
    ban = normalize_postal_code(row['code_postal_ban'])
    brut = normalize_postal_code(row['code_postal_brut'])
    # a missing postal code is considered as a mismatch
    if ban is None or brut is None:
        return False
    return ban == brut

def get_rnb_id(row):
    if not same_postal_code(row):
        return []

    ban_id = row['identifiant_ban']
    if pd.isna(ban_id):
        return []

    with connection.cursor() as cursor:
        cursor.execute(SQL, [ban_id])
        result = cursor.fetchone()
    return result[0] if result[0] is not None else []

def execute(df):
    df_copy = df.copy()
    try:
        df_copy['rnb_id_rappro'] = df_copy.apply(get_rnb_id, axis=1)
    finally:
        # each thread gets its own DB connection: close it so it is not leaked
        connection.close()
    return df_copy


def split_dataframe(df, n):
    # np.array_split() does not return DataFrames anymore since pandas 3.0
    # (DataFrame.swapaxes was removed), so we split the index instead
    return [df.iloc[idx] for idx in np.array_split(np.arange(len(df)), n)]


def process_sub_file(i):
    print(f"processing file {i}")
    if not os.path.exists(f'notebooks/rapprochements/DPE/2026/sub_files_results/dpe-{i}-result.csv'):
        # dtype=str keeps the postal codes and the BAN id exactly as written in the file
        df_sub_file = pd.read_csv(f'notebooks/rapprochements/DPE/2026/sub_files/dpe-{i}.csv', sep=',', dtype=str)
        max_workers = 50
        dfs = split_dataframe(df_sub_file, max_workers)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            results_sub_file = executor.map(execute, dfs)
            df_result = pd.concat(results_sub_file)
            df_result.to_csv(f'notebooks/rapprochements/DPE/2026/sub_files_results/dpe-{i}-result.csv', index=False)

for i in range(100):
    process_sub_file(i)

processing file 0
processing file 1
processing file 2
processing file 3
processing file 4
processing file 5
processing file 6
processing file 7
processing file 8
processing file 9
processing file 10
processing file 11
processing file 12
processing file 13
processing file 14
processing file 15
processing file 16
processing file 17
processing file 18
processing file 19
processing file 20
processing file 21
processing file 22
processing file 23
processing file 24
processing file 25
processing file 26
processing file 27
processing file 28
processing file 29
processing file 30
processing file 31
processing file 32
processing file 33
processing file 34
processing file 35
processing file 36
processing file 37
processing file 38
processing file 39
processing file 40
processing file 41
processing file 42
processing file 43
processing file 44
processing file 45
processing file 46
processing file 47
processing file 48
processing file 49
processing file 50
processing file 51
processing file 52
pro

In [8]:
# aggregation des sous fichiers de résultats en un seul
dfs_result = []

for i in range(100):
    df_result = pd.read_csv(f'notebooks/rapprochements/DPE/2026/sub_files_results/dpe-{i}-result.csv', usecols=['numero_dpe', 'file', 'rnb_id_rappro', 'id_rnb', 'provenance_id_rnb'])
    dfs_result.append(df_result)

df_final = pd.concat(dfs_result)
df_final['rnb_id_rappro'] = df_final['rnb_id_rappro'].apply(eval)


In [ ]:
# sauvegarde des résultats
df_final.to_csv('notebooks/rapprochements/DPE/2026/results_DPE_RNB.csv', index=False)

: 